# Build a RAG Agent 🤖🧠

[30-second quickstart](https://docs.langchain.com/oss/python/langchain/agents)

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG) agent** using vector search, help you understand and implement a basic RAG system step by step:

**Install**

In [1]:
pip install -U langchain langchain_openai

--- Logging error ---
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 177, in emit
    self.console.print(renderable, overflow="ignore", crop=False, style=style)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1673, in print
    extend(render(renderable, render_options))
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1305, in render
    for render_output in iter_render:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 134, in __rich_console__
    for line in lines:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/segment.py", line 249, in split_lines
    for segment in segments:
  File "/Library/

**Environment**

In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv

from colorama import Fore, Style

# Load environment variables from .env file
load_dotenv(override=True)  

# Set OPENAI_API_KEY from environment variable
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded from environment variable.{Style.RESET_ALL}")

OpenAI API key loaded from environment variable.


**Initialize model**

In [3]:
from langchain.chat_models import init_chat_model


# -----------------------------
# LLM Setup
# -----------------------------

# A low-temperature model gives more repeatable demonstrations.
model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0,
    timeout=30,
    max_retries=3,
)

**Data Loading**

In [4]:
!mkdir -p 'data/forge/'
!wget \
  "https://raw.githubusercontent.com/magit/forge/refs/heads/main/CHANGELOG" \
  -O "data/forge/CHANGELOG.txt"

--2026-07-17 14:59:03--  https://raw.githubusercontent.com/magit/forge/refs/heads/main/CHANGELOG
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 25405 (25K) [text/plain]
Saving to: ‘data/forge/CHANGELOG.txt’

data/forge/CHANGELO 100%[===================>]  24,81K  --.-KB/s    in 0,004s  

2026-07-17 14:59:04 (6,79 MB/s) - ‘data/forge/CHANGELOG.txt’ saved [25405/25405]



**Document Loading**

In [5]:
from pathlib import Path
from langchain_core.documents import Document

DATA_DIR = Path("./data/forge/")


def load_text_documents(directory: Path) -> list[Document]:
    """Load every .txt file as a LangChain Document."""

    if not directory.exists():
        raise FileNotFoundError(
            f"Data directory not found: {directory.resolve()}"
        )

    documents: list[Document] = []

    for file_path in directory.glob("*.txt"):
        text = file_path.read_text(encoding="utf-8")

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": file_path.name,
                    "file_path": str(file_path),
                },
            )
        )

    if not documents:
        raise ValueError(
            f"No .txt files were found in {directory.resolve()}"
        )

    return documents


documents = load_text_documents(DATA_DIR)

**Documents Processing**

In [6]:

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
)

chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print(f"Documents loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print(chunks[0].metadata)
print(documents[0].metadata)
print(documents[0].page_content[:300])



Documents loaded: 1
Chunks created: 96
{'source': 'CHANGELOG.txt', 'file_path': 'data/forge/CHANGELOG.txt'}
{'source': 'CHANGELOG.txt', 'file_path': 'data/forge/CHANGELOG.txt'}
# -*- mode: org -*-
* v0.6.7    2026-07-01

- Adjusted to breaking change in ~magit-branch-p~.  f3ad5910

- Fixed a regression when visiting blob or copying its URL.  #858

* v0.6.6    2026-06-01

- Fixed a regression in interaction with Ghub.  #848

- No longer error if ~ssh~ is missing when lookin


**Prompts**

In [7]:
prompt = """You are a grounded research assistant with access to a document search tool.

Use the search_knowledge_base tool whenever the user asks about:
You are a grounded knowledge assistant.

Use only the retrieved context to answer the question.

Rules:
- Cite supporting sources using their filenames in square brackets.
- Do not use outside knowledge.
- Do not invent facts.
- If the answer is not in the context, say so.
- Keep the answer focused on the question,
      
  Retrieved context: {context}
  Question: {question}"""


**utils**

In [8]:

def format_documents(docs: list[Document]) -> str:
    """Format retrieved documents with visible source labels."""

    if not docs:
        return "No relevant context was retrieved."

    formatted = []

    for position, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        year = doc.metadata.get("year", "unknown")
        topic = doc.metadata.get("topic", "unknown")

        formatted.append(
            f"[Source {position}]\n"
            f"File: {source}\n"
            f"Year: {year}\n"
            f"Topic: {topic}\n"
            f"Content: {doc.page_content}"
        )

    return "\n\n".join(formatted)

**create a tool**

In [9]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for relevant documents and return their content."""
    
    retrieved_documents = retriever.invoke(query)

    if not retrieved_documents:
        return (
            "No relevant documents were found."
        )
    return format_documents(retrieved_documents)

**create an agent **

In [10]:
# create an agent
agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[search_knowledge_base],
    system_prompt=prompt,
)

In [11]:
questions = [
    "What pagination-related problem is described in the unreleased "
    "section of the Forge changelog?"
    "What external platform changed the format of label IDs?"
    "What operation could be completely prevented by the pagination bug?"
]

**Run RAG Agent**

In [12]:
from colorama import Fore, Style

# invoke and run agent
question = questions[0]

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
)
answer = result["messages"][-1].content
print(f"{Fore.CYAN}- Question: {question}{Style.RESET_ALL}")
print(f"{Fore.GREEN}- Answer: {answer}{Style.RESET_ALL}")


- Question: What pagination-related problem is described in the unreleased section of the Forge changelog?What external platform changed the format of label IDs?What operation could be completely prevented by the pagination bug?
- Answer: The pagination-related problem described in the unreleased section of the Forge changelog is a pagination bug that could completely prevent pulling API data for certain GitHub repositories.

The external platform that changed the format of label IDs is GitHub.

The operation that could be completely prevented by the pagination bug was pulling API data for certain GitHub repositories.

These points are summarized in the changelog entry from version 0.6.5 (2026-05-01):
- "Fixed a pagination bug, which could completely prevent pulling API data for certain Github repositories. #843"
- "Fixed a regression in v0.5.1, which prevented labeling topics with labels that were created before Github changed the format of label IDs. #784" [CHANGELOG.txt]
